# Ring Resonator Analysis, Part 5: Wafer Map

Building a wafer map traditionally means collecting per-die result files from a directory tree, parsing die coordinates out of filenames or paths, and joining them into a grid. Here the die coordinates come from the `die` tag and the radius comes from the `radius_nm` tag, so the grouping is a single query with no filename parsing.

In the previous notebook we computed the mean FSR per (die, radius) group. Now we aggregate those results into a spatial wafer map so you can see whether the FSR is uniform across the wafer or whether there are systematic gradients.

Because each radius group has its own characteristic FSR, we trigger one wafer map pipeline per (wafer, radius) combination. The spec limits (4.68 to 4.76 nm) are appropriate for R = 20 µm rings. You would adjust `min_output` and `max_output` for other radii.

We reuse `aggregate_die_analyses.py` from the cutback series, pointing it at the `mean` field in each die JSON.

## Setup

In [ ]:
import getpass
from pathlib import Path

import gfhub
from gfhub import nodes
from PIL import Image
from tqdm.auto import tqdm

client = gfhub.Client()
user = getpass.getuser()
print(f"Running as user: {user}")

## Defining the wafer map function

`fsr_wafer_map` reads per-die JSON files, extracts die coordinates and a configurable metric, and renders a two-panel figure:

- **Left panel**: pass/fail map. Green for dies within spec, blue for too low, red for too high, hatched for missing.
- **Right panel**: continuous colour scale showing the actual values, so you can spot spatial gradients.

The function uses masked arrays so that only the relevant cells are drawn in each layer. This avoids colour-stacking artifacts that can occur when overlaying multiple transparent `pcolor` calls.

In [ ]:
def fsr_wafer_map(
    files: list[Path],
    /,
    *,
    output_key: str = "mean",
    min_output: float = 0.0,
    max_output: float = 1.0,
    output_name: str = "wafer_map",
) -> Path:
    """Build a spatial wafer map from per-die FSR JSON results."""
    analyses = {}
    for file in files:
        data = json.loads(file.read_text())
        value = data.get(output_key, np.nan)
        x, y = int(data["die_x"]), int(data["die_y"])
        analyses[(x, y)] = value

    if not analyses:
        msg = "No die analyses found with valid coordinates"
        raise ValueError(msg)

    # Build the wafer grid
    coords = np.array(sorted(analyses.keys()))
    die_xs = np.unique(coords[:, 0])
    die_ys = np.unique(coords[:, 1])
    x_min, x_max = int(die_xs.min()), int(die_xs.max())
    y_min, y_max = int(die_ys.min()), int(die_ys.max())
    nx = x_max - x_min + 1
    ny = y_max - y_min + 1

    # Cell edges at half-integers so cells are centred on integer die coordinates
    x_edges = np.arange(nx + 1) + x_min - 0.5
    y_edges = np.arange(ny + 1) + y_min - 0.5
    X, Y = np.meshgrid(x_edges, y_edges, indexing="ij")

    data_grid = np.full((nx, ny), np.nan)
    for (x, y), value in analyses.items():
        data_grid[x - x_min, y - y_min] = value

    exists = ~np.isnan(data_grid)
    toolow = exists & (data_grid < min_output)
    toohigh = exists & (data_grid > max_output)
    good = exists & ~toolow & ~toohigh
    ones = np.ones((nx, ny))

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4.5))

    # --- Left panel: pass/fail map ---
    # Use masked arrays so only the relevant cells are drawn per layer
    ax0.pcolormesh(X, Y, np.ma.masked_where(~good, ones),
                   cmap=mcolors.ListedColormap(["#00ff00"]), vmin=0, vmax=1, alpha=0.8)
    ax0.pcolormesh(X, Y, np.ma.masked_where(~toolow, ones),
                   cmap=mcolors.ListedColormap(["blue"]), vmin=0, vmax=1, alpha=0.8)
    ax0.pcolormesh(X, Y, np.ma.masked_where(~toohigh, ones),
                   cmap=mcolors.ListedColormap(["red"]), vmin=0, vmax=1, alpha=0.8)
    ax0.pcolor(X, Y, np.ma.masked_where(exists, ones),
               hatch="//", edgecolor="C7", facecolor="none", linewidth=0)

    ax0.plot([], [], "s", c="#00ff00", alpha=0.8, label="good")
    ax0.plot([], [], "s", c="blue", alpha=0.8, label=f"too low [<{min_output:.2f}]")
    ax0.plot([], [], "s", c="red", alpha=0.8, label=f"too high [>{max_output:.2f}]")
    ax0.legend(loc="upper right", fontsize=7)

    # --- Right panel: continuous values ---
    ax1.pcolormesh(X, Y, np.ma.masked_where(~exists, data_grid),
                   vmin=min_output, vmax=max_output)
    ax1.pcolor(X, Y, np.ma.masked_where(exists, ones),
               hatch="//", edgecolor="C7", facecolor="none", linewidth=0)

    # Value labels and axis formatting for both panels
    for ax in (ax0, ax1):
        for i in range(nx):
            for j in range(ny):
                v = data_grid[i, j]
                if not np.isnan(v):
                    ax.text(i + x_min, j + y_min, f"{v:.2f}",
                            ha="center", va="center", color="black", fontsize=8)
        ax.set_xticks(die_xs)
        ax.set_xticks(0.5 * (die_xs[1:] + die_xs[:-1]), minor=True)
        ax.set_yticks(die_ys)
        ax.set_yticks(0.5 * (die_ys[1:] + die_ys[:-1]), minor=True)
        ax.grid(visible=True, which="minor", ls=":")
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("die x")
        ax.set_ylabel("die y")

    fig.suptitle(output_key)
    plt.tight_layout()

    output_path = files[0].parent / f"{output_name}.png"
    plt.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.close()

    return output_path


func_def = gfhub.Function(
    fsr_wafer_map,
    dependencies={
        "json": "import json",
        "numpy": "import numpy as np",
        "matplotlib": [
            "import matplotlib.pyplot as plt",
            "import matplotlib.colors as mcolors",
        ],
    },
)
client.add_function(func_def)
print("fsr_wafer_map uploaded.")

## Creating the pipeline

The pipeline receives a group of die-analysis JSON files (one per die) and produces a single wafer map PNG. The `output_key` is `"mean"` to match the field written by `fsr_for_radius_within_die` in notebook 4. The spec limits (4.68 to 4.76 nm) are appropriate for R = 20 µm rings, adjust `min_output` and `max_output` for other radii.

`find_common_tags` extracts the tags shared by all input files so the output is tagged with the wafer, project, and radius without any die-specific tags leaking through.

In [ ]:
p = gfhub.Pipeline()

p.trigger = nodes.on_manual_trigger()
p.load_file = nodes.load()
p.load_tags = nodes.load_tags()
p += p.trigger >> p.load_file
p += p.trigger >> p.load_tags

p.find_common_tags = nodes.function(function="find_common_tags")
p += p.load_tags >> p.find_common_tags

p.aggregate = nodes.function(
    function="fsr_wafer_map",
    kwargs={
        "output_key": "mean",
        "min_output": 4.68,
        "max_output": 4.76,
    },
)
p += p.load_file >> p.aggregate

p.save = nodes.save()
p += p.aggregate >> p.save[0]
p += p.find_common_tags >> p.save[1]

confirmation = client.add_pipeline(name="wafer_fsr_aggregation", schema=p)
print(f"Pipeline ready: {client.pipeline_url(confirmation['id'])}")

## Trigger per (wafer, radius) group

We restrict the query to `radius_nm:20000` (R = 20 µm) for this wafer map. To generate maps for other radii, change this tag value. The `groupby("wafer")` then separates results by wafer without needing a separate directory per wafer or a manifest file.

In [ ]:
entries = client.query_files(
    name="die_fsr_*.json",
    tags=["project:tutorial_rings", user, "wafer", "radius_nm:20000"],
).groupby("wafer")

print(f"Found {len(entries)} wafer(s)")

job_ids = []
for wafer_tag, group in tqdm(entries.items()):
    print(f"Processing {wafer_tag}: {len(group)} dies")
    input_ids = [f["id"] for f in group]
    triggered = client.trigger_pipeline("wafer_fsr_aggregation", input_ids)
    job_ids.extend(triggered["job_ids"])

print(f"Triggered {len(job_ids)} wafer analysis jobs")

In [ ]:
jobs = client.wait_for_jobs(job_ids)
print(f"All jobs complete. Statuses: {set(j['status'] for j in jobs)}")

## View the wafer maps

In [ ]:
wafer_maps = client.query_files(
    name="wafer_map.png",
    tags=["project:tutorial_rings", user],
)
print(f"Found {len(wafer_maps)} wafer maps")

for wm in wafer_maps:
    img = Image.open(client.download_file(wm["id"]))
    display(img)